# 01 — Testy i wykresy wartosci `pi` (spike-and-slab prior na gamma)

Notebook bada, jak prior `pi = P(gamma = 1)` wplywa na to, **ktore czynniki model uznaje
za rozniaace kohorty**. Wszystkie metryki licza sie wylacznie z **wyjsc dopasowanego
modelu** (`E[gamma]`, `E[delta]`, ELBO) oraz **etykiet kohort** — bez ground-truth, bo
FACTM uczy sie nienadzorowanie i prawdziwych Z/W/mu zwykle nie znamy.

## Kontrakt danych (co produkuje pierwsza komorka)
- `views` — obiekt `Views` z ustawionym `cohorts` (>= 1 widok prosty, Normal).
- `K` — liczba czynnikow do dopasowania.
- (wyprowadzane automatycznie) `cohort_names`, `codes`, `C`, `factors`.

Podmien wylacznie sekcje oznaczona `>>> PODMIEN ... <<<` w pierwszej komorce.

## Komorka 1 — wczytanie danych

In [ ]:
# =====================================================================
#  KOMORKA 1 — WCZYTANIE DANYCH
# =====================================================================
import os, sys, warnings
# Znajdz katalog zawierajacy pakiet `src` (dziala z final_notebooks/ i z roota repo).
_p = os.path.abspath('')
for _ in range(4):
    if os.path.isdir(os.path.join(_p, 'src')):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        break
    _p = os.path.dirname(_p)
warnings.filterwarnings('ignore')

import numpy as np
from src.views import Views


def make_synthetic_cohort_data(C=4, K=4, n_active=2, n_per_cohort=120,
                               dims=(20, 15), sigma=0.6, noise=1.0,
                               mu_scale=1.9, block_sparse_W=False, seed=0):
    """Domyslne dane DEMO (do podmiany na wlasne).

    C kohort o roznych profilach mu_c w przestrzeni K czynnikow; tylko pierwsze
    `n_active` czynnikow rozni kohorty (reszta wspolna -- "cicha"). Widoki powstaja
    jako Y_m = Z W_m^T + szum. Zwraca (views, codes); NIE zwraca ground-truth --
    cala analiza korzysta tylko z `views` i etykiet kohort.
    """
    rng = np.random.default_rng(seed)
    mu = np.zeros((C, K))
    mu[:, :n_active] = rng.normal(scale=mu_scale, size=(C, n_active))
    codes = np.repeat(np.arange(C), n_per_cohort)
    Z = np.array([rng.normal(mu[c], sigma) for c in codes])
    Ys = []
    for d in dims:
        if block_sparse_W:
            W = np.zeros((d, K))                              # prawdziwe zera poza blokiem
            for k, idx in enumerate(np.array_split(np.arange(d), K)):
                W[idx, k] = rng.normal(loc=1.5, scale=0.25, size=len(idx))  # blok cech -> czynnik k
        else:
            W = rng.normal(size=(d, K))
        Ys.append(Z @ W.T + rng.normal(scale=noise, size=(Z.shape[0], d)))
    cohorts = np.array([f'c{c}' for c in codes])
    return Views.from_list(Ys, cohorts=cohorts), codes


# >>>>>>>>>>>>>>>>>>>>>>>  PODMIEN TE SEKCJE NA SWOJE DANE  >>>>>>>>>>>>>>>>>>>>>>>
# Wymagane: `views` (Views z cohorts) oraz `K`.
views, _ = make_synthetic_cohort_data(C=4, K=4, n_active=2, seed=0)
K = 4
# <<<<<<<<<<<<<<<<<<<<<<<  KONIEC SEKCJI DO PODMIANY  <<<<<<<<<<<<<<<<<<<<<<<<<<<<<

# Wyprowadzenie etykiet kohort z danych (dziala tez dla Twoich danych):
assert views.cohorts is not None, "views musi miec ustawione cohorts (etykieta kohorty per probka)."
cohort_names, codes = np.unique(views.cohorts, return_inverse=True)
C = len(cohort_names)
factors = [f'Z{k}' for k in range(K)]
print(f"N = {views.N} probek | widoki (liczba cech): {[v.D for v in views.simple]}")
print(f"C = {C} kohort: {list(cohort_names)} | K = {K} czynnikow do dopasowania")

## Setup — model i helpery

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import FACTModel
from src.enums import Likelihood, WPrior, ZPrior
from src.model_config import CohortPriorConfig, ModelConfig, SimpleViewConfig

def fit_cohort(views, K, pi=0.5, max_iter=100, seed=0):
    """Dopasuj CohortFACTM: WPrior.ARD_SS + ZPrior.COHORT, prior spike-and-slab `pi`."""
    cfg = ModelConfig(
        simple_view_configs=[SimpleViewConfig(likelihood=Likelihood.NORMAL, w_prior=WPrior.ARD_SS)
                             for _ in range(views.num_simple)],
        structured_view_configs=[],
        z_priors=[ZPrior.COHORT] * K,
        cohort_prior_config=CohortPriorConfig(pi=pi),
    )
    m = FACTModel(views=views, K=K, model_config=cfg, seed=seed)
    m.fit(max_iter=max_iter, pretrain=True, elbo_tres=0.0)
    return m

def cohort_gamma_delta(model, K):
    """Macierze (C, K): E[gamma] (prawdopodobienstwo aktywacji kohortowej) i E[delta]
    (przesuniecie sredniej). Wiersze w kolejnosci np.unique(cohorts) == `cohort_names`."""
    priors = model.fa.node_z.z_priors
    assert priors[0].__class__.__name__ == 'CohortZPrior', "model nie ma cohort-prior"
    E_g = np.column_stack([priors[k].E_gamma for k in range(K)])
    E_d = np.column_stack([priors[k].E_delta for k in range(K)])
    return E_g, E_d

## Test 1 — diagnostyka gamma przy domyslnym `pi`

Pojedyncze dopasowanie przy `pi=0.5`: ktore czynniki aktywuja sie jako kohortowe
(`E[gamma] -> 1`), a ktore pozostaja ciche (`E[gamma] -> 0`). Heatmapy pokazuja
aktywacje `E[gamma]` oraz pelny profil kohortowy `E[gamma*delta]`.

In [ ]:
PI_DEFAULT = 0.5
m = fit_cohort(views, K=K, pi=PI_DEFAULT, seed=0)
E_g, E_d = cohort_gamma_delta(m, K)

tab = pd.DataFrame({
    'mean E[gamma]': E_g.mean(axis=0).round(3),
    'max E[gamma]':  E_g.max(axis=0).round(3),
    'max|E[delta]|': np.abs(E_d).max(axis=0).round(3),
}, index=factors)
print(f"pi = {PI_DEFAULT} | ELBO = {float(m.elbo_sequence[-1]):.1f}")
print(tab.to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 0.8 + 0.45 * C))
sns.heatmap(E_g, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1,
            xticklabels=factors, yticklabels=cohort_names, ax=axes[0])
axes[0].set_title('E[gamma] — aktywacja kohortowa (kohorta x czynnik)')
sns.heatmap(E_g * E_d, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            xticklabels=factors, yticklabels=cohort_names, ax=axes[1])
axes[1].set_title('E[gamma*delta] — profil kohortowy')
plt.tight_layout(); plt.show()

## Test 2 — sweep `pi`: aktywacja gamma, ELBO, liczba aktywnych czynnikow

Dla siatki `pi` dopasowujemy model i zbieramy `mean E[gamma]` per czynnik, liczbe
czynnikow uznanych za aktywne (`mean E[gamma] > prog`) oraz ELBO. Oczekiwanie: silne
czynniki kohortowe aktywne w calym zakresie, slabe/ciche zaczynaja sie falszywie
aktywowac dopiero przy duzym `pi`.

In [ ]:
pi_grid = [0.05, 0.1, 0.2, 0.3, 0.5, 0.7, 0.9]
ACTIVE_THR = 0.5   # czynnik "aktywny" gdy mean E[gamma] > prog

rows, gamma_by_pi = [], {}
for pi in pi_grid:
    mp = fit_cohort(views, K=K, pi=pi, seed=0)
    Eg, _ = cohort_gamma_delta(mp, K)
    g_mean = Eg.mean(axis=0)
    gamma_by_pi[pi] = g_mean
    rows.append({'pi': pi,
                 **{f'gamma_{factors[k]}': g_mean[k] for k in range(K)},
                 'n_active': int((g_mean > ACTIVE_THR).sum()),
                 'elbo': float(mp.elbo_sequence[-1])})
df_pi = pd.DataFrame(rows)
print(df_pi.round(3).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for k in range(K):
    axes[0].plot(df_pi['pi'], df_pi[f'gamma_{factors[k]}'], 'o-', linewidth=2, label=factors[k])
axes[0].axhline(ACTIVE_THR, color='gray', ls='--', alpha=0.6, label=f'prog {ACTIVE_THR}')
axes[0].set_xlabel('pi = P(gamma=1)'); axes[0].set_ylabel('mean E[gamma]')
axes[0].set_title('Aktywacja gamma per czynnik vs pi'); axes[0].set_ylim(-0.05, 1.05)
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

axes[1].plot(df_pi['pi'], df_pi['n_active'], 'o-', color='C3', linewidth=2)
axes[1].set_xlabel('pi'); axes[1].set_ylabel(f'liczba czynnikow (mean E[gamma] > {ACTIVE_THR})')
axes[1].set_title('Liczba aktywnych czynnikow vs pi'); axes[1].grid(alpha=0.3)
axes[1].set_ylim(-0.2, K + 0.2)

axes[2].plot(df_pi['pi'], df_pi['elbo'], 'o-', color='C2', linewidth=2)
axes[2].set_xlabel('pi'); axes[2].set_ylabel('ELBO'); axes[2].set_title('ELBO vs pi')
axes[2].grid(alpha=0.3)
plt.tight_layout(); plt.show()

mat = np.array([gamma_by_pi[pi] for pi in pi_grid]).T   # (K, len(pi))
fig, ax = plt.subplots(figsize=(1.5 + 1.1 * len(pi_grid), 0.8 + 0.5 * K))
sns.heatmap(mat, annot=True, fmt='.2f', cmap='RdYlGn', vmin=0, vmax=1,
            xticklabels=[f'{p:g}' for p in pi_grid], yticklabels=factors,
            cbar_kws={'label': 'mean E[gamma]'}, ax=ax)
ax.set_xlabel('pi'); ax.set_ylabel('czynnik')
ax.set_title('Mapa aktywacji gamma w zaleznosci od pi')
plt.tight_layout(); plt.show()

## Test 3 — jak `pi` ksztaltuje profil kohortowy `E[gamma*delta]`

Dla kilku wartosci `pi` porownujemy aktywacje `E[gamma]` (gorny wiersz) z pelnym
profilem kohortowym `E[gamma*delta]` (dolny wiersz). Maly `pi` = konserwatywny
(mniej aktywacji), duzy `pi` = liberalny (wiecej, ryzyko falszywych aktywacji).

In [ ]:
pi_show = [0.1, 0.5, 0.9]
fig, axes = plt.subplots(2, len(pi_show), figsize=(4.2 * len(pi_show), 1.5 + 0.8 * C))
for j, pi in enumerate(pi_show):
    mp = fit_cohort(views, K=K, pi=pi, seed=0)
    Eg, Ed = cohort_gamma_delta(mp, K)
    sns.heatmap(Eg, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1,
                xticklabels=factors, yticklabels=cohort_names, ax=axes[0, j], cbar=False)
    axes[0, j].set_title(f'E[gamma]  (pi={pi})')
    sns.heatmap(Eg * Ed, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                xticklabels=factors, yticklabels=cohort_names, ax=axes[1, j], cbar=False)
    axes[1, j].set_title(f'E[gamma*delta]  (pi={pi})')
plt.tight_layout(); plt.show()

### Interpretacja
- Czynniki o silnym sygnale kohortowym maja `mean E[gamma]` blisko 1 niezaleznie od `pi`.
- Czynniki ciche rosna z `pi` — przy duzym `pi` model "dopisuje" strukture, ktorej nie ma.
- Sweet spot zwykle ~`0.3-0.5`: aktywne czynniki wykryte, ciche jeszcze stlumione.